# 01 — Data wrangling: §4.1 accounting consistency (kernel-conformant audit)

**Status.** Notebook 1 of the `cbase2/` pipeline, conformed to the canonical
kernel as a **differential audit**. It replaced the self-contained conformance
port of 2026-09-17 by direction of the repository owner (the port is preserved
in git history at `6d66265`); ADR-0011 authorises edits to this one file.

**One kernel.** The §4.1 transformation is *implemented once*, in the root
package: `BeyondHulten.generate_data` (`src/core/accounting.jl`), bound to a
`Data` object by `assemble_data`, and extended to the open economy by
`recalibrate_open` / `retained_dataset` (`src/core/calibration.jl`). This
notebook therefore does **not** re-implement the pipeline as production code.
It recomputes the transformation *independently* and asserts that the kernel
reproduces it — a differential audit, plus the traps that the audit exists to
catch.

**Input.** `data/I-O_DE2019_formatiert.csv` (German 2019 symmetric use table,
Destatis format), SHA-256 printed in the setup cell. The byte-identical copy
under `cbase2/data_raw/` (same SHA-256, `e9c32997…`) is the frozen provenance
of the original pipeline.

**Outputs.** `output/` — the repository's gitignored diagnostic directory
(`check_repo.jl` check 7, ADR-0007). Written here: the §4.1 calibration table,
the GDP reconciliation, the dataset-variant coverage, and the contract audit
matrix. Regenerate rather than commit.

*Artifact contract:* the replaced self-contained version wrote the seven
`cbase2/data_processed/` artifacts that `02_accounting_consistency.ipynb`
consumes. This audit does not: it reads the canonical `data/` table and writes
to `output/`, so those tracked artifacts are no longer regenerated here (they
remain unchanged on disk, and a regeneration of the old code path reproduces
them byte-identically — measured 2026-09-17).

**Conventions kept from the original notebook.** Named schema constants, the
proportional import allocation as a *stated* assumption, value added kept as an
explicit composite (total VA is never relabelled "labour"), and one assertion
block that fails loudly rather than silently.

In [ ]:
# ===========================================================================
# Setup -- environment, paths, provenance.
#
# Paths derive from @__DIR__ (never a machine-local absolute path); the input
# table is hashed so a run is checkable after the fact.
# ===========================================================================

# find_root(start): walk upward from `start` until the BeyondHulten
# Project.toml is found; that directory is the repository root. Keeps the
# notebook cwd-independent (no machine-local absolute paths).
function find_root(start)
    d = abspath(start)
    while d != dirname(d)
        p = joinpath(d, "Project.toml")
        if isfile(p) && occursin("name = \"BeyondHulten\"", read(p, String))
            return d
        end
        d = dirname(d)
    end
    error("BeyondHulten project root not found above $start")
end

# Self-activation guard: on a normal machine this activates the repo project;
# a harness running with `--project=.` already resolves the package and skips it.
if Base.find_package("BeyondHulten") === nothing
    import Pkg
    Pkg.activate(find_root(@__DIR__))
end
using BeyondHulten, CSV, DataFrames, LinearAlgebra, Statistics, SHA

ROOT   = find_root(@__DIR__)
INPUT  = joinpath(ROOT, "data", "I-O_DE2019_formatiert.csv")
OUTDIR = joinpath(ROOT, "output")            # gitignored diagnostic output
mkpath(OUTDIR)

INPUT_SHA = bytes2hex(sha256(read(INPUT)))
println("repo root     : ", ROOT)
println("input table   : ", INPUT)
println("input SHA-256 : ", INPUT_SHA)
println("output dir    : ", OUTDIR)

## Step 0 — Load the table and declare the schema by name

The schema is declared as **named constants**, and every row is located **by
label** — never by a hard-coded table position. The final-demand block in
particular is resolved with the kernel's own `final_demand_columns`
(`_FD_COLUMN_NAMES`, `src/core/accounting.jl`), so the same notebook works on a
rebuilt retained-sector table, where the sector block is shorter and the
positions shift.

The reader contract is checked rather than assumed: the table is read directly
with the documented Destatis conventions (`;` delimiter, `,` decimal,
`-`/`x` missing) and the result must equal the table carried inside the `Data`
object that `read_data` returns.

In [ ]:
# ===========================================================================
# Step 0 -- load the raw table and declare the schema by name.
# ===========================================================================
N = 71                                    # modelled sectors (symmetric table)
SEC = 2:(N + 1)                           # sector columns (supplier rows / user cols)

# Canonical load path: `read_data` carries the raw table in `data.io`.
data = read_data("I-O_DE2019_formatiert.csv"; datadir = ROOT)
io   = data.io

# --- Reader contract: the documented Destatis conventions, read independently
io_check = CSV.read(INPUT, DataFrames.DataFrame;
                    delim = ';', decimal = ',', missingstring = ["-", "x"])
rename!(io_check, Symbol(names(io_check)[1]) => :Sektoren)
io_check.Sektoren = replace.(io_check.Sektoren, r"^\s+" => "")   # leading blanks
io_check = coalesce.(io_check, 0)                               # "-" / "x" -> 0
@assert Matrix{Float64}(io_check[1:N, SEC]) == Matrix{Float64}(io[1:N, SEC]) "reader contract: the Destatis conventions must reproduce data.io"
@assert all(String.(io_check.Sektoren[1:N]) .== String.(io.Sektoren[1:N])) "reader contract: sector labels"

# --- Final-demand block located BY NAME, never by table position
FD_NAMES = ["Konsumausgaben der privaten Haushalte im Inland",
            "Konsumausgaben der privaten Organisationen o.E.",
            "Konsumausgaben des Staates",
            "Anlageinvestitionen f.Ausrüstungen u.sonst.Anlagen",
            "Anlageinvestitionen für Bauten",
            "Vorratsveränderungen und Nettozugang an Wertsachen",
            "Exporte"]
FD_any = [findfirst(==(n), names(io)) for n in FD_NAMES]
@assert all(FD_any .!== nothing) "final-demand block must resolve by label"
FD = Int.(FD_any)
@assert length(FD) == 7 && issorted(FD)
# Cross-check against the kernel's own locator (unexported internal: qualified).
if isdefined(BeyondHulten, :final_demand_columns)
    @assert FD == BeyondHulten.final_demand_columns(io) "notebook and kernel must locate the same final-demand columns"
    println("final-demand locator agrees with BeyondHulten.final_demand_columns")
end

# --- Import / product-tax rows located BY LABEL as well
r_imp = findfirst(==("Verwendung der Importe"), io.Sektoren)
r_tx  = findfirst(==("Gütersteuern abzüglich Gütersubventionen"), io.Sektoren)
@assert r_imp !== nothing && r_tx !== nothing "import/product-tax rows not found by name"

println("table: nrow=", nrow(io), " ncol=", ncol(io), " sectors=", N)
println("final-demand columns (by label)  : ", FD)
println("import row / product-tax row      : ", r_imp, " / ", r_tx)

### Canary: the off-by-one import indexing trap

The original pipeline's most expensive silent bug: slicing the row block with
`io[row, 2:end]` **first** and then indexing that slice with the original
sector range `2:72` shifts every import/tax value by one column, producing a
spurious GDP residual that hides the true one. The cell below reads the import
row both ways and asserts that they differ — the trap is live, so the safe
(labelled, direct) form is the one used everywhere else in this notebook.

In [ ]:
# ===========================================================================
# Canary -- off-by-one import indexing (documented trap; must stay detectable).
# ===========================================================================
imp_row_ok  = Matrix{Float64}(io[r_imp:r_imp, SEC])[:]              # direct: correct
io_sliced   = io[:, 2:end]                                         # slice the columns first ...
imp_row_bug = Matrix{Float64}(io_sliced[r_imp:r_imp, SEC])[:]        # ... then re-index with SEC: SHIFTED

shift = maximum(abs.(imp_row_ok .- imp_row_bug))
@assert !isapprox(imp_row_ok, imp_row_bug) "off-by-one canary went silent: the shifted read should differ"
println("off-by-one canary: max|correct - shifted| = ", round(shift; sigdigits = 6),
        " EUR m  (total imported intermediates = ", round(sum(imp_row_ok)), ")")

## Step 1 — Domestic vs imported uses; basic vs purchaser prices

Imports are reported **by using sector** only, so the domestic intermediate
matrix $Z^{D}$ allocates them proportionally at the user level (a *stated*
assumption, not a data fact):

$$Z^{D}_{s,u} = Z_{s,u} \cdot \frac{\text{domestic intermediate into } u}{\text{total intermediate into } u}$$

Users whose recorded imports exceed their total intermediate use are clamped to
a fully import-dependent row. From $Z^{D}$ we form the conditional
input-share matrices in Baqaee–Farhi orientation (rows = using sector,
columns = supplying sector), each row summing to 1 (or exactly 0 where the
domestic content is nil, which is stated, not hidden as `NaN`):

- $\Omega^{D}$ — domestic conditional shares (audit / domestic incidence);
- $\Omega^{raw}$ — total conditional shares (the equilibrium *technology*
  matrix and the consumption-share calibration use these).

In [ ]:
# ===========================================================================
# Step 1 -- independent domestic/import split and conditional share matrices.
# ===========================================================================
imp_inter   = Matrix{Float64}(io[r_imp:r_imp, SEC])[:]   # imported intermediates, by user
imp_final   = Matrix{Float64}(io[r_imp:r_imp, FD])[:]    # imported final demand, by category
imptx_inter = Matrix{Float64}(io[r_tx:r_tx, SEC])[:]     # product taxes, by user
imptx_final = Matrix{Float64}(io[r_tx:r_tx, FD])[:]      # product taxes, by category

Z = Matrix{Float64}(io[1:N, SEC])                        # Z[s,u]: supplied by s to user u

inter_into_user = vec(sum(Z; dims = 1))                  # total intermediate use per user
dom_remainder   = inter_into_user .- imp_inter
clamped_users   = findall(<(0), dom_remainder)           # imports > intermediate use
dom_into_user   = max.(dom_remainder, 0.0)               # stated clamp (not an error)
Z_dom = Z .* (dom_into_user' ./ max.(inter_into_user, 1e-12)')

# Conditional share matrices: rows = users, so the divisor must broadcast DOWN
# the rows -- a Vector, i.e. a column vector, on the right.
coltot  = vec(sum(Z_dom; dims = 1))
Ω_dom_i = permutedims(Z_dom) ./ max.(coltot, 1e-12)      # Ω_dom[u,s]
Ω_dom_i[coltot .<= 0, :] .= 0.0                          # nil domestic content -> explicit 0 row
Ω_raw_i = permutedims(Z) ./ vec(sum(Z; dims = 1))        # Ω_raw[u,s]

rows_dom = vec(sum(Ω_dom_i; dims = 2))
rows_raw = vec(sum(Ω_raw_i; dims = 2))
@assert all(isapprox.(rows_dom, 1.0; rtol = 1e-12) .| (coltot .<= 0)) "Ω_dom rows must sum to 1 (or 0)"
@assert all(isapprox.(rows_raw, 1.0; rtol = 1e-12)) "Ω_raw rows must sum to 1"

println("using sectors with imports > intermediate use : ", length(clamped_users), " -> ", clamped_users)
for u in clamped_users
    println("   sector ", lpad(u, 2), ": ", io.Sektoren[u])
end
println("Ω_dom row sums : min=", minimum(rows_dom), "  max=", maximum(rows_dom),
        "   all-zero rows=", count(<(1e-12), rows_dom))
println("Ω_raw row sums : min=", minimum(rows_raw), "  max=", maximum(rows_raw))

### Canary: Ω orientation (the row-vector trap)

Normalising with a **row** vector (`permutedims(Z_dom) ./ totals'`) divides by
the *supplying-sector* index instead of the *using-sector* one, so columns — not
rows — are normalised and the share rows no longer sum to 1. The error is silent
in the sense that nothing throws; it simply produces a technology matrix that
violates the CES price-index contract at every $\theta \neq 1$. Measured below.

In [ ]:
# ===========================================================================
# Canary -- Ω orientation: a row-vector divisor normalises the WRONG axis.
# ===========================================================================
Ω_dom_trap = permutedims(Z_dom) ./ max.(coltot, 1e-12)'   # row vector: divides columns
rows_trap  = vec(sum(Ω_dom_trap; dims = 2))
@assert !all(isapprox.(rows_trap, 1.0; rtol = 1e-12)) "orientation canary went silent"
println("orientation canary: row sums under the WRONG axis -> min=", minimum(rows_trap),
        "  max=", maximum(rows_trap))
println("                    row sums under the CORRECT divisor -> min=", minimum(rows_dom),
        "  max=", maximum(rows_dom))

### Differential check against the kernel

`generate_data` recomputes Steps 1–3 internally. The independent leg above must
reproduce `data.Ω` and `data.Ω_raw` **to machine precision** — not approximately:
the two paths do the same arithmetic on the same table, so any nonzero
difference is a defect on one side or the other, and there is no tolerance to
hide behind.

In [ ]:
# ===========================================================================
# Differential check -- independent leg vs the kernel's generate_data.
# ===========================================================================
dΩ  = maximum(abs.(Ω_dom_i .- data.Ω))
dΩr = maximum(abs.(Ω_raw_i .- data.Ω_raw))
@assert dΩ == 0.0  "Ω_dom disagrees with the kernel: $dΩ"
@assert dΩr == 0.0 "Ω_raw disagrees with the kernel: $dΩr"

kernel_zero = findall(<(1e-12), vec(sum(data.Ω; dims = 2)))
@assert kernel_zero == clamped_users "the kernel's zero-share rows must be exactly the clamped users"
@assert all(data.Ω[clamped_users, :] .== 0.0)
println("Ω_dom / Ω_raw agree with the kernel exactly (0.0)")
println("kernel Ω_dom zero rows = ", kernel_zero, "  (identical to the clamped set)")

## Step 2 — Decompose value added; never relabel it "labour"

Value added is read as its four components and asserted to sum to gross value
added. The kernel keeps the composite and exposes the labour role separately:
`factor_share = gva / gross_output_basic` is a **composite** value-added weight,
while `value_added_components.wage` carries compensation of employees. The
notebook checks both, and the labour-compensation share of gross output that the
original notebook published as `wage_share_gross_output` is recovered from the
kernel's component frame (it is not a `Data` field — by design).

In [ ]:
# ===========================================================================
# Step 2 -- value-added decomposition (independent leg vs kernel).
# ===========================================================================
rowvec(name) = Matrix{Float64}(io[findfirst(==(name), io.Sektoren):findfirst(==(name), io.Sektoren), SEC])[:]

gva     = rowvec("Bruttowertschöpfung")                        # gross value added, basic prices
wage    = rowvec("Arbeitnehmerentgelt im Inland")              # compensation of employees
othertx = rowvec("Sonst.Produktionsabgaben abzgl. sonst.Subventionen")
dep     = rowvec("Abschreibungen")                             # consumption of fixed capital
netop   = rowvec("Nettobetriebsüberschuss")                    # net operating surplus
prodval = rowvec("Produktionswert")                            # gross output, basic prices

va_total = wage .+ othertx .+ dep .+ netop
va_gap   = maximum(abs.(va_total .- gva))
@assert isapprox(va_total, gva; rtol = 1e-9) "VA components must sum to gross value added"

factor_share_i = gva ./ prodval                                # COMPOSITE VA share
wage_share_go  = wage ./ prodval                               # labour compensation / gross output

cols = Set(names(data.value_added_components))
d_wage = maximum(abs.(wage .- data.value_added_components.wage))
d_gva  = maximum(abs.(gva  .- data.value_added_components.gross_value_added))
d_fs   = maximum(abs.(factor_share_i .- data.factor_share))
@assert d_wage == 0.0 && d_gva == 0.0 && d_fs == 0.0 "VA decomposition disagrees with the kernel"
@assert va_gap == 0.0
println("VA identity |components - GVA| max = ", va_gap)
println("value-added components frame     : ", sort(collect(cols)))
println("mean factor_share (composite)    = ", round(mean(factor_share_i); digits = 4),
        "   mean wage_share (labour/GO) = ", round(mean(wage_share_go); digits = 4))
println("(factor_share is the COMPOSITE VA weight -- not pure labour.)")

## Step 3 — Domar weights, absorption, and the §4.1 bridge

Standard Domar weights are gross-output shares, $\lambda_s = y_s / \mathrm{GDP}_P$,
always positive. The deprecated Leontief-inverse form
$(\,\mathrm{diag}(\text{factor\_share})' (I - \mathrm{diag}(1-\text{factor\_share})\,\Omega)^{-1}\,)'$
is the one piece of the legacy pipeline that must **not** come back: with the
correct *domestic* $\Omega$ it is not singular (measured conditioning below) —
it is simply the wrong object, because it prices intermediate rounds that the
open-economy accounting has already removed. The canary reports the gap.

The model-facing bridge is the domestic final-demand vector from the kernel's
proportional split of the purchaser-price categories.

In [ ]:
# ===========================================================================
# Step 3 -- Domar weights, deprecated form, domestic final-demand bridge.
# ===========================================================================
GDP_P = sum(gva)
GDP_I = sum(wage) + sum(othertx) + sum(dep) + sum(netop)
@assert GDP_P ≈ GDP_I "production-side GDP must equal income-side GDP"

λ_i          = prodval ./ GDP_P                # standard Domar weights (always positive)
labor_share_i = λ_i .* factor_share_i          # composite VA weight, model's "labor_share"
@assert all(>(0), λ_i) "Domar weights must be positive"
@assert isapprox(sum(labor_share_i), 1.0; rtol = 1e-12) "Σ labor_share must be 1 (baseline income unit)"

d_lam = maximum(abs.(λ_i .- data.λ))
d_lab = maximum(abs.(labor_share_i .- data.labor_share))
@assert d_lam == 0.0 && d_lab == 0.0 "Domar weights disagree with the kernel"

# --- Deprecated Leontief-inverse Domar form: wrong, not singular
M_dep = I - Diagonal(1.0 .- factor_share_i) * Ω_dom_i
λ_dep = (factor_share_i' * inv(M_dep))'
println("deprecated Leontief form: cond(I - diag(1-fs)*Ω_dom) = ", round(cond(M_dep); digits = 4))
println("   Σ λ_deprecated = ", round(sum(λ_dep); digits = 4), "  vs   Σ λ_standard = ", round(sum(λ_i); digits = 4))
println("   max |λ_deprecated - λ_standard| = ", round(maximum(abs.(λ_dep .- λ_i)); digits = 4),
        "   (deprecated: use prodval / GDP_P)")

## Step 4 — GDP three-side reconciliation

Production and income agree exactly by construction (the VA components sum to
gross value added). The expenditure side is built from the purchaser-price final
demand net of the recorded imports and product taxes. The residual is a
**genuine valuation gap of the raw table**, not a bug and not to be "fixed":
final-demand columns are recorded at purchaser prices whose recorded
import/tax content does not close the basic-price expenditure identity. The
kernel asserts only a 10 % gross-error guard; the notebook reports the measured
residual. (Independent confirmation of the value recorded for the frozen
pipeline: ≈5 % on the full table.)

In [ ]:
# ===========================================================================
# Step 4 -- GDP three-side reconciliation.
# ===========================================================================
fd_tot       = Matrix{Float64}(io[1:N, FD])
fd_imp_total = sum(imp_final)
fd_tax_total = sum(imptx_final)
fd_purch     = sum(fd_tot)
GDP_E        = fd_purch - fd_imp_total - fd_tax_total
resid_PE     = abs(GDP_P - GDP_E) / GDP_P

d_gdp = maximum(abs.([GDP_P, GDP_I, GDP_E] .- [data.gdp_production, data.gdp_income, data.gdp_expenditure]))
@assert d_gdp == 0.0 "GDP aggregates disagree with the kernel"
@assert resid_PE < 0.10 "production vs expenditure discrepancy > 10% (gross accounting error)"

println("GDP production (Σ GVA)            = ", round(GDP_P; digits = 1))
println("GDP income     (VA components)    = ", round(GDP_I; digits = 1))
println("GDP expenditure (purch - imp - tx)= ", round(GDP_E; digits = 1))
println("  final demand at purchaser prices= ", round(fd_purch; digits = 1),
        "   imported = ", round(fd_imp_total; digits = 1),
        "   product taxes = ", round(fd_tax_total; digits = 1))
println("residual |P - E| / P              = ", round(100 * resid_PE; digits = 3), " %  (raw-table valuation gap)")
println("identity DGP production == income : ", round(GDP_P - GDP_I; digits = 6))

## Step 5 — The §4.1 bridge: domestic final demand, and the negative-category guard

The domestic final-demand vector is the model-facing bridge between the table
and the absorption blocks. One subtlety worth stating: the **inventories**
category has a *negative* total in this table. A naive proportional split
(`total × max(ratio, 0)`) then flips sign twice and produces a large, wrong
domestic demand for that category; the kernel's guard
(`category total > 0 ? split : 0`) encodes "no domestic content for a
non-positive category" instead. The canary measures the difference.

In [ ]:
# ===========================================================================
# Step 5 -- domestic final demand, with the non-positive-category guard.
# ===========================================================================
function domestic_split(fd_tot, imp_final, imptx_final)
    dom = similar(fd_tot)
    for k in axes(fd_tot, 2)
        cat_total = sum(fd_tot[:, k])
        domfrac   = cat_total > 0 ? (cat_total - imp_final[k] - imptx_final[k]) / cat_total : 0.0
        dom[:, k] = fd_tot[:, k] .* max(domfrac, 0.0)
    end
    return dom
end
dom_kernel = domestic_split(fd_tot, imp_final, imptx_final)

# Naive variant WITHOUT the non-positive guard (documented trap)
dom_naive = similar(fd_tot)
for k in axes(fd_tot, 2)
    cat_total = sum(fd_tot[:, k])
    domfrac   = (cat_total - imp_final[k] - imptx_final[k]) / cat_total
    dom_naive[:, k] = fd_tot[:, k] .* max(domfrac, 0.0)
end
bad_cats = findall(<=(0), [sum(fd_tot[:, k]) for k in axes(fd_tot, 2)])
println("final-demand categories with non-positive totals: ", bad_cats,
        " (values ", round.([sum(fd_tot[:, k]) for k in bad_cats]; digits = 0), ")")
println("guard canary: max |guarded - naive| domestic final demand = ",
        round(maximum(abs.(dom_kernel .- dom_naive)); digits = 1), " EUR m")

domestic_fd_i = vec(sum(dom_kernel; dims = 2))
d_fd = maximum(abs.(domestic_fd_i .- data.domestic_final_demand))
@assert d_fd == 0.0 "domestic final demand disagrees with the kernel"
@assert maximum(abs.(imp_inter ./ prodval .- data.import_share)) == 0.0

println("Σ domestic final demand = ", round(sum(domestic_fd_i); digits = 1),
        "   mean import share of gross output = ", round(mean(imp_inter ./ prodval); digits = 4),
        "   max = ", round(maximum(imp_inter ./ prodval); digits = 4),
        " (sector ", argmax(imp_inter ./ prodval), ")")

## Step 6 — Sector-exclusion variants: rebuild from the raw table

The three dataset variants (`full`, `70s`, `reduced`) are rebuilt by
`retained_dataset`, which slices the **raw table** and re-runs the whole
transformation on the slice — never slices already-derived matrices. Slicing
derived matrices leaves each retained user's shares summing to
$1 - (\text{share bought from the dropped sectors})$, which breaks the CES
price-index contract across any elasticity continuation through
$\theta = 1$ and destroys the income normalization. The assertions that catch
this are the kernel's loud ones: every $\Omega^{raw}$ row sums to 1 and
$\sum \text{labor\_share} = 1$, for **every** variant.

*Variant status at the calibration level* (`registry/preregistration.toml`,
A-bill record): only `full` (71) and `70s` are calibrated; the `70s` variant
carries one microscopic negative residual ($-2.4\times10^{-6}$, clamped and
documented) and the **`reduced` variant is deferred** — its domestic-bill
identity gap is $\approx 7.7\times10^{-3}$, which fails the kernel's
`recalibrate_open` guard. The table below therefore reports coverage and the
normalization contracts for all three, but the notebook does **not** claim a
usable calibration for `reduced`.

In [ ]:
# ===========================================================================
# Step 6 -- dataset variants: coverage and normalization contracts.
# ===========================================================================
variant_rows = NamedTuple[]
for name in ["full", "70s", "reduced"]
    drops = DATASET_VARIANTS[name]
    d     = retained_dataset(data, drops)
    cov   = dataset_coverage(data, drops)
    rowsums = vec(sum(d.Ω_raw; dims = 2))
    @assert isapprox(sum(d.labor_share), 1.0; rtol = 1e-10) "$name: Σ labor_share must be 1"
    @assert all(isapprox.(rowsums, 1.0; rtol = 1e-10)) "$name: Ω_raw rows must sum to 1"
    push!(variant_rows, (variant = name, sectors = length(d.λ),
                         drops = join(sort(drops), ","),
                         gross_output_kept_pct = cov.gross_share_kept,
                         value_added_kept_pct  = cov.va_share_kept,
                         household_fd_kept_pct = cov.fd_share_kept,
                         Sigma_labor_share     = round(sum(d.labor_share); digits = 12)))
end
variants = DataFrame(variant_rows)
println(variants)

## Step 7 — Calibration snapshot (reported, not pinned)

`recalibrate_open` fills the v3 open-economy blocks from the same raw table. The
numbers below are **reported, not asserted**: they are a calibration *record*,
and they move when the kernel's calibration changes. Two contracts are checked
structurally — the saving rate lies in $[0,1)$ and the CPI weights sum to 1 —
because those are the kernel's own claims.

*State of the kernel at this writing:* the calibration carries the committed
"exact domestic-**A-bill**" fix (`Data.A_bill` / `Data.M_int`,
`recalibrate_open`): the intermediate demand is charged with the table's
domestic-bill row (73) instead of the purchaser-price bill
$(1-\text{fs})\lambda \equiv A + \text{imports} + \text{taxes}$, which removes
the baseline clamp and puts the imported/taxed intermediate content into the
external account explicitly. `registry/preregistration.toml` records the
re-anchored rate ($s = 0.1199$ full-71, $0.1285$ for 70s, intermediate imports
$M_{int} = 0.2214$ of GDP) — the snapshot below is an independent reproduction
of that record rather than a copy of it. The cell reads the A-bill fields when
they are present, so it also runs against a pre-fix kernel.

In [ ]:
# ===========================================================================
# Step 7 -- open-economy calibration snapshot (reported, not pinned).
# ===========================================================================
cal = recalibrate_open(data)
@assert 0.0 <= cal.saving_rate < 1.0 "saving rate must lie in [0, 1)"
@assert isapprox(sum(cal.consumption_share), 1.0; rtol = 1e-12) "CPI weights must sum to 1"

has_abill = :A_bill in fieldnames(Data)
println("saving rate s        = ", cal.saving_rate)
println("government (tau0)    = ", sum(cal.gov_demand))
println("exports              = ", sum(cal.exports_demand))
println("investment           = ", sum(cal.exo_demand))
println("baseline household c0= ", sum(cal.household_baseline))
println("A-bill fields present: ", has_abill)
if has_abill
    println("Σ A_bill (domestic intermediate bill) = ", sum(cal.A_bill))
    println("Σ M_int  (intermediate import leak)   = ", sum(cal.M_int))
end

## Step 8 — Persist the §4.1 calibration table and the audit record

`ROADMAP.md` §4.1 asks for a published calibration table containing at least
intermediate-input shares, value-added shares, final-demand shares, import
shares and Domar weights. The table below carries those per sector together
with the GDP sides and a contract-audit matrix, written to `output/` — the
gitignored diagnostic directory — so the record survives the session without
being committed as a generated artifact.

In [ ]:
# ===========================================================================
# Step 8 -- persist the §4.1 calibration table and the contract audit matrix.
# ===========================================================================
cal_table = DataFrame(
    sector                     = String.(io.Sektoren[1:N]),
    gross_output_basic         = prodval,
    value_added_basic          = gva,
    factor_share               = factor_share_i,           # composite VA weight
    wage_share_gross_output    = wage_share_go,            # labour compensation / GO
    lambda_domar               = λ_i,
    labor_share_composite      = labor_share_i,
    import_share_intermediate  = imp_inter ./ prodval,
    domestic_final_demand      = domestic_fd_i,
    consumption_share_gross_go = data.consumption_share_gross_output,
)
CSV.write(joinpath(OUTDIR, "AC_data_wrangling_calibration.csv"), cal_table)

recon = DataFrame(quantity = ["GDP_production", "GDP_income", "GDP_expenditure",
                              "final_demand_purchaser", "imported_final", "product_taxes_final",
                              "residual_PE_pct"],
                  value = [GDP_P, GDP_I, GDP_E, fd_purch, fd_imp_total, fd_tax_total,
                           100 * resid_PE])
CSV.write(joinpath(OUTDIR, "AC_gdp_reconciliation.csv"), recon)
CSV.write(joinpath(OUTDIR, "AC_variants_coverage.csv"), variants)

audit = DataFrame(contract = ["Ω_dom vs kernel", "Ω_raw vs kernel",
                              "VA components vs kernel", "factor_share vs kernel",
                              "Domar λ vs kernel", "labor_share vs kernel",
                              "domestic final demand vs kernel", "import_share vs kernel",
                              "GDP aggregates vs kernel",
                              "reader contract (destatis conventions)"],
                  notebook = [dΩ, dΩr, d_wage, d_fs, d_lam, d_lab, d_fd,
                              maximum(abs.(imp_inter ./ prodval .- data.import_share)), d_gdp, 0.0],
                  tolerance = fill(0.0, 10))
audit[!, :pass] = audit[!, :notebook] .<= audit[!, :tolerance]
CSV.write(joinpath(OUTDIR, "AC_contract_audit.csv"), audit)

println("written to output/:")
println("  AC_data_wrangling_calibration.csv  (", nrow(cal_table), " sectors)")
println("  AC_gdp_reconciliation.csv")
println("  AC_variants_coverage.csv")
println("  AC_contract_audit.csv")

## Step 9 — Final gate

Every invariant of the notebook re-asserted in one place, the artifact list
verified, and the one-line call to reproduce the record.

In [ ]:
# ===========================================================================
# Step 9 -- final gate: re-assert every contract, verify the artifacts.
# ===========================================================================
@assert isapprox(va_total, gva; rtol = 1e-9)                        "VA = GVA"
@assert all(isapprox.(vec(sum(Ω_dom_i; dims = 2)), 1.0; rtol = 1e-12) .| (coltot .<= 0))
@assert all(isapprox.(vec(sum(Ω_raw_i; dims = 2)), 1.0; rtol = 1e-12))
@assert all(isapprox.(vec(sum(data.Ω_raw; dims = 2)), 1.0; rtol = 1e-12))
@assert GDP_P ≈ GDP_I
@assert resid_PE < 0.10
@assert dΩ == 0.0 && dΩr == 0.0 && d_fs == 0.0 && d_lam == 0.0 && d_fd == 0.0 && d_gdp == 0.0
@assert all(audit[!, :pass]) "contract audit must be clean"
@assert nrow(variants) == 3

for f in ["AC_data_wrangling_calibration.csv", "AC_gdp_reconciliation.csv",
          "AC_variants_coverage.csv", "AC_contract_audit.csv"]
    @assert isfile(joinpath(OUTDIR, f)) "missing artifact: $f"
end
println("All 01_data_wrangling assertions passed (", nrow(audit), " contracts, 0 tolerance).")
println()
println("Reproduce:  jupyter lab cbase2/01_data_wrangling.ipynb   (kernel julia-1.12)")
println("Headless:   julia --project=. -e 'using Pkg; Pkg.test()'   # the kernel-side regression suite")
println("Input SHA-256: ", INPUT_SHA)